In [0]:
dbutils.widgets.text("catalog_name", "allianz_coe")
dbutils.widgets.text("gold_schema_name", "ab_gold_test")
dbutils.widgets.text("vault_schema_name", "silver")
dbutils.widgets.text("control_schema_name", "audit_control")
dbutils.widgets.text("watermark_table_name", "application_watermark")

catalog_name = dbutils.widgets.get("catalog_name")
gold_schema_name = dbutils.widgets.get("gold_schema_name")
vault_schema = dbutils.widgets.get("vault_schema_name")
control_schema_name = dbutils.widgets.get("control_schema_name")
watermark_table_name = dbutils.widgets.get("watermark_table_name")

In [0]:
FACT_LEAD_CFG = {
    "name": "fact_lead",
    "target_table": f"{catalog_name}.{gold_schema_name}.fact_lead",

    "stage_sql": f"""
        WITH wm AS (
            SELECT
                COALESCE(
                    MAX(watermark),
                    TIMESTAMP('1900-01-01 00:00:00')
                ) AS watermark
            FROM {catalog_name}.{control_schema_name}.{watermark_table_name}
            WHERE table_name = '{catalog_name}.{gold_schema_name}.fact_lead'
        ),
        src AS (
            SELECT
                COALESCE(dp.person_sk, -1) AS person_sk,
                COALESCE(dd.date_sk, -1) AS date_sk,
                COALESCE(dm.marketing_sk, -1) AS marketing_sk,
                hl.lead_id AS lead_id,
                dp.person_id AS person_id,
                sl.interested_level AS interested_level,
                sl.person_score AS person_score,
                sl.person_status AS person_status,
                sl.converted_date AS lead_creation_ts,
                'ETL_SYSTEM' AS created_by,
                current_timestamp() AS created_ts,
                to_timestamp(sl.load_date) AS load_ts,
                to_timestamp(sl.load_date) AS event_ts
            FROM {catalog_name}.{vault_schema}.hub_lead hl
        
            LEFT JOIN {catalog_name}.{vault_schema}.sat_lead sl
                ON hl.lead_hash_key = sl.lead_hash_key
        
            LEFT JOIN {catalog_name}.{vault_schema}.link_person_lead lpl
                ON lpl.lead_hash_key = hl.lead_hash_key
        
            LEFT JOIN {catalog_name}.{vault_schema}.hub_person hp
                ON hp.person_hash_key = lpl.person_hash_key
        
            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_person dp
                ON dp.person_id = hp.person_id
        
            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_date dd
                ON to_date(dd.full_date, 'yyyy/MM/dd') = to_date(sl.converted_date)
        
            LEFT JOIN {catalog_name}.{vault_schema}.link_person_marketing_preference lpmp
                ON lpmp.person_hash_key = hp.person_hash_key
        
            LEFT JOIN {catalog_name}.{vault_schema}.hub_marketing_preference hmp
                ON hmp.marketing_preference_hash_key = lpmp.marketing_preference_hash_key
        
            LEFT JOIN {catalog_name}.{vault_schema}.link_person_marketing_engagement lpme
                ON lpme.person_hash_key = hp.person_hash_key
        
            LEFT JOIN {catalog_name}.{vault_schema}.hub_marketing_engagement hme
                ON hme.marketing_engagement_hash_key = lpme.marketing_engagement_hash_key
        
            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_marketing dm
                ON dm.marketing_preference_id = hmp.marketing_preference_id
            AND dm.marketing_engagement_id = hme.marketing_engagement_id
        
            WHERE to_timestamp(sl.load_date) > (SELECT watermark FROM wm)
        ),
        dedup AS (
                    SELECT
                        src.*,
                        ROW_NUMBER() OVER (
                            PARTITION BY lead_id
                            ORDER BY event_ts DESC
                        ) AS rn
                    FROM src
                )
        SELECT
            person_sk,
            date_sk,
            marketing_sk,
            lead_id,
            person_id,
            interested_level,
            person_score,
            person_status,
            lead_creation_ts,
            created_by,
            created_ts,
            load_ts
        FROM dedup
        WHERE rn = 1
    """,

    "increment_keys": ["lead_id"],
}

In [0]:
FACT_QUOTE_CFG = {
    "name": "fact_quote",
    "target_table": f"{catalog_name}.{gold_schema_name}.fact_quote",

    "stage_sql": f"""
        WITH wm AS (
            SELECT
                COALESCE(
                    MAX(watermark),
                    TIMESTAMP('1900-01-01 00:00:00')
                ) AS watermark
            FROM {catalog_name}.{control_schema_name}.{watermark_table_name}
            WHERE table_name = '{catalog_name}.{gold_schema_name}.fact_quote'
        ),
        src AS (
            SELECT
                COALESCE(dp.person_sk, -1) AS person_sk,
                COALESCE(dc.customer_sk, -1) AS customer_sk,
                COALESCE(da.account_sk, -1) AS account_sk,
                COALESCE(dm.marketing_sk, -1) AS marketing_sk,
                COALESCE(dd.date_sk, -1) AS date_sk,
                COALESCE(dcam.campaign_sk, -1) AS campaign_sk,
                COALESCE(dcha.channel_sk, -1) AS channel_sk,
                COALESCE(db.broker_sk, -1) AS broker_sk,
                COALESCE(doi.insured_object_sk, -1) AS insured_object_sk,

                hq.quote_id AS quote_id,
                dp.person_id AS person_id,

                sq.quote_number AS quote_number,
                sq.quote_status AS quote_status,
                sq.gross_revenue AS quote_gross_revenue_amt,
                sq.net_revenue AS quote_net_revenue_amt,
                sq.renewal_amt_current_period AS quote_renewal_current_period_amt,
                sq.renewal_amt_next_period AS quote_renewal_next_period_amt,
                sq.quoted_premium AS quoted_premium,
                sq.quote_date AS quote_date,
                sq.quote_month_name AS quote_month_name,
                sq.risk_score AS risk_score,
                sq.policy_complexity AS policy_complexity,
                sq.uw_approval_type AS uw_approval_type,
                sq.rejection_reason AS rejection_reason,

                'ETL_SYSTEM' AS created_by,
                current_timestamp() AS created_ts,
                to_timestamp(sq.load_date) AS load_ts,
                to_timestamp(sq.load_date) AS event_ts

            FROM {catalog_name}.{vault_schema}.hub_quote hq

            LEFT JOIN {catalog_name}.{vault_schema}.sat_quote sq
                ON hq.quote_hash_key = sq.quote_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.link_quote_person rqp
                ON rqp.quote_hash_key = hq.quote_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_person hp
                ON hp.person_hash_key = rqp.person_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_person dp
                ON dp.person_id = hp.person_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_customer_person rcp
                ON rcp.person_hash_key = hp.person_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_customer hc
                ON hc.customer_hash_key = rcp.customer_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_customer dc
                ON dc.customer_id = hc.customer_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_person_account lpa
                ON lpa.person_hash_key = hp.person_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_account ha
                ON ha.account_hash_key = lpa.account_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_account da
                ON da.account_id = ha.account_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_person_marketing_preference rpmp
                ON rpmp.person_hash_key = hp.person_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_marketing_preference hmp
                ON hmp.marketing_preference_hash_key = rpmp.marketing_preference_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.link_person_marketing_engagement rpme
                ON rpme.person_hash_key = hp.person_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_marketing_engagement hme
                ON hme.marketing_engagement_hash_key = rpme.marketing_engagement_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_marketing dm
                ON dm.marketing_preference_id = hmp.marketing_preference_id
                AND dm.marketing_engagement_id = hme.marketing_engagement_id

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_date dd
                ON to_date(dd.full_date, 'yyyy/MM/dd') = to_date(sq.quote_date)

            LEFT JOIN {catalog_name}.{vault_schema}.link_person_campaign rpcam
                ON rpcam.person_hash_key = hp.person_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_campaign hcam
                ON hcam.campaign_hash_key = rpcam.campaign_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_campaign dcam
                ON dcam.campaign_id = hcam.campaign_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_quote_channel rqcha
                ON rqcha.quote_hash_key = hq.quote_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_channel hcha
                ON hcha.channel_hash_key = rqcha.channel_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_channel dcha
                ON dcha.channel_id = hcha.channel_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_quote_broker rqb
                ON rqb.quote_hash_key = hq.quote_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_broker hb
                ON hb.broker_hash_key = rqb.broker_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_broker db
                ON db.agent_id = hb.agent_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_policy_quote rpq
                ON rpq.quote_hash_key = hq.quote_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_policy hpo
                ON hpo.policy_hash_key = rpq.policy_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.link_policy_insured_object rpio
                ON rpio.policy_hash_key = hpo.policy_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_insured_object hio
                ON hio.insured_object_hash_key = rpio.insured_object_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_insured_object doi
                ON doi.insured_object_id = hio.insured_object_id

            WHERE sq.load_date > (SELECT watermark FROM wm)
        ),
        dedup AS (
            SELECT
                src.*,
                ROW_NUMBER() OVER (
                    PARTITION BY quote_id
                    ORDER BY event_ts DESC
                ) AS rn
            FROM src
        )

        SELECT
            person_sk,
            customer_sk,
            account_sk,
            marketing_sk,
            date_sk,
            campaign_sk,
            channel_sk,
            broker_sk,
            insured_object_sk,

            quote_id,
            person_id,
            quote_number,
            quote_status,
            quote_gross_revenue_amt,
            quote_net_revenue_amt,
            quote_renewal_current_period_amt,
            quote_renewal_next_period_amt,
            quoted_premium,
            quote_date,
            quote_month_name,
            risk_score,
            policy_complexity,
            uw_approval_type,
            rejection_reason,
            created_by,
            created_ts,
            load_ts
        FROM dedup
        WHERE rn = 1;
    """,

    "increment_keys": ["quote_id"],
}

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
FACT_POLICY_CFG = {
    "name": "fact_policy",
    "target_table": f"{catalog_name}.{gold_schema_name}.fact_policy",

    "stage_sql": f"""
        WITH wm AS (
            SELECT
                COALESCE(
                    MAX(watermark),
                    TIMESTAMP('1900-01-01 00:00:00')
                ) AS watermark
            FROM {catalog_name}.{control_schema_name}.{watermark_table_name}
            WHERE table_name = '{catalog_name}.{gold_schema_name}.fact_policy'
        ),
        src AS (
            SELECT
                COALESCE(dpo.policy_sk, -1) AS policy_sk,
                COALESCE(dp.person_sk, -1) AS person_sk,
                COALESCE(dc.customer_sk, -1) AS customer_sk,
                COALESCE(da.account_sk, -1) AS account_sk,
                COALESCE(dd.date_sk, -1) AS date_sk,
                COALESCE(dm.marketing_sk, -1) AS marketing_sk,
                COALESCE(dch.channel_sk, -1) AS channel_sk,
                COALESCE(db.broker_sk, -1) AS broker_sk,
                COALESCE(dcl.claim_sk, -1) AS claim_sk,
                COALESCE(dov.override_sk, -1) AS override_sk,
                COALESCE(dio.insured_object_sk, -1) AS insured_object_sk,
                dp.person_id AS person_id,
                sp.number_of_active_claim AS active_claims_number,
                sp.number_of_previous_claim AS previous_claims_number,
                sp.declined_claims AS declined_claims_number,
                sp.gross_revenue AS policy_gross_revenue_amt,
                sp.net_revenue AS policy_net_revenue_amt,
                sp.renewal_amount_current_period AS policy_renewal_current_period_amt,
                sp.renewal_amount_next_period AS policy_renewal_next_period_amt,
                sp.policy_base_premium AS policy_base_premium,
                sp.gross_written_premium AS gross_written_premium,
                sp.earned_premium AS earned_premium,
                sp.incurred_but_not_reported AS incurred_but_not_reported,
                sp.operating_expenses AS operating_expenses,
                sp.administrative_expenses AS administrative_expenses,
                sp.profit_margin AS profit_margin,
                sp.taxes_and_levies AS taxes_and_levies,
                sp.amount_approved AS amt_approved,
                sp.ceded_premium AS ceded_premium,
                sp.commission_paid AS commission_paid,
                sp.ceded_commission AS ceded_commission,
                sp.exposure_amount AS exposure_amt,
                sp.investment_income AS investment_income,
                sp.underwriting_cycle_time_in_days AS underwriting_cycle_time_in_days,
                sp.underwriting_expenses AS underwriting_expenses,
                sp.transaction_date AS transaction_date,
                sp.record_type AS record_type,
                sp.discount AS discount,
                sp.override_commission AS override_commission,
                sp.partial_recovery_percentage AS partial_recovery_percentage, 
                'ETL_SYSTEM'                   AS created_by,
                current_timestamp()            AS created_ts,  
                to_timestamp(sp.load_date)  AS load_ts,
                to_timestamp(sp.load_date) AS event_ts
            FROM {catalog_name}.{vault_schema}.hub_policy hp

            LEFT JOIN {catalog_name}.{vault_schema}.sat_policy sp
                ON hp.policy_hash_key = sp.policy_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_policy dpo
                ON dpo.policy_id = hp.policy_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_policy_customer lpc
                ON lpc.policy_hash_key = hp.policy_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_customer hc
                ON hc.customer_hash_key = lpc.customer_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_customer dc
                ON dc.customer_id = hc.customer_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_customer_person lcp
                ON lcp.customer_hash_key = hc.customer_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_person hper
                ON hper.person_hash_key = lcp.person_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_person dp
                ON dp.person_id = hper.person_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_person_account lpa
                ON lpa.person_hash_key = hper.person_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_account ha
                ON ha.account_hash_key = lpa.account_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_account da
                ON da.account_id = ha.account_id

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_date dd
                ON to_date(dd.full_date, 'yyyy/MM/dd') = to_date(sp.policy_issue_date)

            LEFT JOIN {catalog_name}.{vault_schema}.link_person_marketing_preference lpmp
                ON lpmp.person_hash_key = hper.person_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_marketing_preference hmp
                ON hmp.marketing_preference_hash_key = lpmp.marketing_preference_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.link_person_marketing_engagement lpme
                ON lpme.person_hash_key = hper.person_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_marketing_engagement hme
                ON hme.marketing_engagement_hash_key = lpme.marketing_engagement_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_marketing dm
                ON dm.marketing_preference_id = hmp.marketing_preference_id
            AND dm.marketing_engagement_id = hme.marketing_engagement_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_policy_channel lpch
                ON lpch.policy_hash_key = hp.policy_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_channel hch
                ON hch.channel_hash_key = lpch.channel_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_channel dch
                ON dch.channel_id = hch.channel_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_policy_broker lpb
                ON lpb.policy_hash_key = hp.policy_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_broker hb
                ON hb.broker_hash_key = lpb.broker_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_broker db
                ON db.agent_id = hb.agent_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_claim_policy lclp
                ON lclp.policy_hash_key = hp.policy_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_claim hcl
                ON hcl.claim_hash_key = lclp.claim_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_claim dcl
                ON dcl.claim_id = hcl.claim_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_policy_override lpo
                ON lpo.policy_hash_key = hp.policy_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_override hov
                ON hov.override_hash_key = lpo.override_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_override dov
                ON dov.override_id = hov.override_id

            LEFT JOIN {catalog_name}.{vault_schema}.link_policy_insured_object lpio
                ON lpio.policy_hash_key = hp.policy_hash_key

            LEFT JOIN {catalog_name}.{vault_schema}.hub_insured_object hio
                ON hio.insured_object_hash_key = lpio.insured_object_hash_key

            LEFT JOIN {catalog_name}.{gold_schema_name}.dim_insured_object dio
                ON dio.insured_object_id = hio.insured_object_id

            WHERE to_timestamp(sp.load_date) > (SELECT watermark FROM wm)
        ),
        dedup AS (
                            SELECT
                                src.*,
                                ROW_NUMBER() OVER (
                                    PARTITION BY policy_sk
                                    ORDER BY event_ts DESC
                                ) AS rn
                            FROM src
                        )

        SELECT
            policy_sk,
            person_sk,
            customer_sk,
            account_sk,
            date_sk,
            marketing_sk,
            channel_sk,
            broker_sk,
            claim_sk,
            override_sk,
            insured_object_sk,
            person_id,
            active_claims_number,
            previous_claims_number,
            declined_claims_number,
            policy_gross_revenue_amt,
            policy_net_revenue_amt,
            policy_renewal_current_period_amt,
            policy_renewal_next_period_amt,
            policy_base_premium,
            gross_written_premium,
            earned_premium,
            incurred_but_not_reported,
            operating_expenses,
            administrative_expenses,
            profit_margin,
            taxes_and_levies,
            amt_approved,
            ceded_premium,
            commission_paid,
            ceded_commission,
            exposure_amt,
            investment_income,
            underwriting_cycle_time_in_days,
            underwriting_expenses,
            transaction_date,
            record_type,
            discount,
            override_commission,
            partial_recovery_percentage,
            created_by,
            created_ts, 
            load_ts
        FROM dedup
        WHERE rn = 1

    """,

    "increment_keys": ["policy_sk"],
}

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:720)
	at com.data

In [0]:
FACT_COMPLAINT_CFG = {
    "name": "fact_complaint",
    "target_table": f"{catalog_name}.{gold_schema_name}.fact_complaint",

    "stage_sql": f"""
                WITH wm AS (
                SELECT
                        COALESCE(
                        MAX(watermark),
                TIMESTAMP('1900-01-01 00:00:00')
                        ) AS watermark
                FROM {catalog_name}.{control_schema_name}.{watermark_table_name}
                WHERE table_name = '{catalog_name}.{gold_schema_name}.fact_complaint'
                ),

                src AS (
                SELECT
                        COALESCE(dpe.person_sk, -1) AS person_sk,
                        COALESCE(dcu.customer_sk, -1) AS customer_sk,
                        COALESCE(dr.regulation_sk, -1) AS regulation_sk,
                        COALESCE(dc.channel_sk, -1) AS channel_sk,
                        COALESCE(dd.date_sk, -1) AS date_sk,
                        COALESCE(doi.insured_object_sk, -1) AS insured_object_sk,

                        hco.complaint_id AS complaint_id,
                        dpe.person_id AS person_id,

                        sco.complaint_date AS complaint_date,
                        sco.complaint_acknowledgement_date AS complaint_acknowledgement_date,
                        sco.complaint_resolved_date AS complaint_resolved_date,
                        sco.complaint_upheld_status AS complaint_upheld_status,
                        sco.is_financial_ombudsman_service_referral AS is_financial_ombudsman_service_referral,
                        sco.complaint_driver AS complaint_driver,
                        sco.complaint_channel AS complaint_channel,
                        sco.compensation_amount AS compensation_amt,
                        sco.complaint_status AS complaint_status,
                        sco.insurance_category AS insurance_category,

                        -- NEW COLUMNS FROM UPDATED S2T
                        sco.complaint_feedback AS complaint_feedback,
                        sco.customer_complaint_satisfaction_score AS customer_complaint_satisfaction_score,

                        'ETL_SYSTEM' AS created_by,
                        current_timestamp() AS created_ts,
                        current_timestamp() AS load_ts,

                        to_timestamp(sco.load_date) AS event_ts

                FROM {catalog_name}.{vault_schema}.hub_complaint hco

                LEFT JOIN {catalog_name}.{vault_schema}.sat_complaint sco
                ON hco.complaint_hash_key = sco.complaint_hash_key

                LEFT JOIN {catalog_name}.{vault_schema}.link_complaint_policy rcp
                ON rcp.complaint_hash_key = hco.complaint_hash_key

                LEFT JOIN {catalog_name}.{vault_schema}.hub_policy hp
                ON hp.policy_hash_key = rcp.policy_hash_key

                LEFT JOIN {catalog_name}.{vault_schema}.link_policy_customer rpcu
                ON rpcu.policy_hash_key = hp.policy_hash_key

                LEFT JOIN {catalog_name}.{vault_schema}.hub_customer hcu
                ON hcu.customer_hash_key = rpcu.customer_hash_key

                LEFT JOIN {catalog_name}.{gold_schema_name}.dim_customer dcu
                ON dcu.customer_id = hcu.customer_id

                LEFT JOIN {catalog_name}.{vault_schema}.link_customer_person rcup
                ON rcup.customer_hash_key = hcu.customer_hash_key

                LEFT JOIN {catalog_name}.{vault_schema}.hub_person hpe
                ON hpe.person_hash_key = rcup.person_hash_key

                LEFT JOIN {catalog_name}.{gold_schema_name}.dim_person dpe
                ON dpe.person_id = hpe.person_id

                LEFT JOIN {catalog_name}.{vault_schema}.link_complaint_regulation rpcr
                ON rpcr.complaint_hash_key = hco.complaint_hash_key

                LEFT JOIN {catalog_name}.{vault_schema}.hub_regulation hr
                ON hr.regulation_hash_key = rpcr.regulation_hash_key

                LEFT JOIN {catalog_name}.{gold_schema_name}.dim_regulation dr
                ON dr.regulation_id = hr.regulation_id

                LEFT JOIN {catalog_name}.{vault_schema}.link_policy_channel rpch
                ON rpch.policy_hash_key = hp.policy_hash_key

                LEFT JOIN {catalog_name}.{vault_schema}.hub_channel hch
                ON hch.channel_hash_key = rpch.channel_hash_key

                LEFT JOIN {catalog_name}.{gold_schema_name}.dim_channel dc
                ON dc.channel_id = hch.channel_id

                LEFT JOIN {catalog_name}.{gold_schema_name}.dim_date dd
                ON to_date(dd.full_date, 'yyyy/MM/dd') = to_date(sco.complaint_date)

                LEFT JOIN {catalog_name}.{vault_schema}.link_policy_insured_object lio
                ON lio.policy_hash_key = hp.policy_hash_key

                LEFT JOIN {catalog_name}.{vault_schema}.hub_insured_object hi
                ON hi.insured_object_hash_key = lio.insured_object_hash_key

                LEFT JOIN {catalog_name}.{gold_schema_name}.dim_insured_object doi
                ON doi.insured_object_id = hi.insured_object_id

                WHERE sco.load_date > (SELECT watermark FROM wm)
                ),

                dedup AS (
                        SELECT
                                src.*,
                                ROW_NUMBER() OVER (
                                PARTITION BY complaint_id
                                ORDER BY event_ts DESC
                                ) AS rn
                        FROM src
                )

                SELECT
                        person_sk,
                        customer_sk,
                        regulation_sk,
                        channel_sk,
                        date_sk,
                        insured_object_sk,
                        person_id,
                        complaint_id,
                        complaint_date,
                        complaint_acknowledgement_date,
                        complaint_resolved_date,
                        complaint_upheld_status,
                        is_financial_ombudsman_service_referral,
                        complaint_driver,
                        complaint_channel,
                        compensation_amt,
                        complaint_status,
                        insurance_category,

                        complaint_feedback,
                        customer_complaint_satisfaction_score,

                        created_by,
                        created_ts,
                        load_ts
                FROM dedup
                WHERE rn = 1
        """,

    "increment_keys": [
        "complaint_id"
    ]
}